# Electrode Map Generator

This notebook maps electrode contacts to PCB or headstage channels.

The workflow:

1. Select electrode layout DXF files.
2. Select component mapping JSON files.
3. Select assembly connection CSV files.
4. Generate a complete channel map.
5. Export the result as CSV.

The generated channel map has shape (4 x N), with rows containing:

1. Electrode X-coordinate in micrometres
2. Electrode Y-coordinate in micrometres
3. Destination channel
4. Source electrode instance

In [1]:
from collections import defaultdict
import tempfile
from pathlib import Path

import ezdxf
from IPython.display import display
import ipywidgets as widgets

import utils
from assembly_routing import AssemblyRouter, ComponentDefinition


In [2]:
# Optional: inspect the layers in an example DXF file.
example_dxf = Path(
    "../electrode_map_example/example1/50um_pitch_compatible_with_FPC.dxf"
)
doc = ezdxf.readfile(example_dxf)
layers = sorted({entity.dxf.layer for entity in doc.modelspace()})
print("Available DXF layers:")
print(layers)


Available DXF layers:
['L2D0_etching2', 'L3D0_etching3', 'electrode', 'etching', 'metalization', 'polyimide', 'polyimide_etch_borders']


In [17]:
# Name of the DXF layer containing electrode contours.
# Set to None when the layer is unknown.
layer_name = "electrode"

# Used only when layer_name is None. The parser removes the final contour,
# which is assumed to be a hook loop, then selects the requested contours.
num_contours = 64

In [13]:
def uploader_with_filename(uploader):
    output = widgets.Output()

    def on_upload(change):
        new_val = change["new"]
        old_val = change["old"]

        # Keep "None" mutually exclusive with file selections.
        if None in new_val and len(new_val) > 1:
            if None not in old_val:
                uploader.value = (None,)
            else:
                uploader.value = tuple(p for p in new_val if p is not None)
            return

        with output:
            output.clear_output()
            filenames = [
                p.relative_to(mapping_dir).as_posix()
                for p in new_val
                if p is not None
            ]

            if filenames:
                print("Chosen file(s):", ", ".join(filenames))
            else:
                print("No file chosen")

    uploader.observe(on_upload, names="value")
    return widgets.VBox([uploader, output])

In [ ]:
mapping_dir = Path("modular_mapping_files")


def make_file_selector(extension, description):
    files = (
        sorted(
            (
                p
                for p in mapping_dir.rglob("*")
                if p.is_file() and p.suffix.lower() == extension.lower()
            ),
            key=lambda p: p.relative_to(mapping_dir).as_posix().lower(),
        )
        if mapping_dir.exists()
        else []
    )
    options = [("None", None)] + [
        (f.relative_to(mapping_dir).as_posix(), f) for f in files
    ]
    return widgets.SelectMultiple(
        options=options,
        value=(None,),
        rows=min(max(len(options), 2), 6),
        description=description,
        style={"description_width": "initial"},
        layout=widgets.Layout(width="450px"),
    )


dxf_chooser = make_file_selector(".dxf", "Select electrode DXF files")
electrode_chooser = make_file_selector(".json", "Select electrode mapping JSON")
flex_pcb_chooser = make_file_selector(".json", "Select flex PCB mapping JSON")
rigid_pcb_chooser = make_file_selector(".json", "Select rigid PCB mapping JSON")
headstage_chooser = make_file_selector(".json", "Select headstage mapping JSON")
assembly_chooser = make_file_selector(".csv", "Select assembly CSV file")

display(uploader_with_filename(dxf_chooser))
display(uploader_with_filename(electrode_chooser))
display(uploader_with_filename(flex_pcb_chooser))
display(uploader_with_filename(rigid_pcb_chooser))
display(uploader_with_filename(headstage_chooser))
display(uploader_with_filename(assembly_chooser))

In [15]:
run_button = widgets.Button(
    description="Generate Channel Map",
    button_style="success",
)
output = widgets.Output()

state = {"channel_map": None, "dxf_file": None}

def get_selected_files(selector_widget):
    """Return the selected file Paths as a list (ignoring None)."""
    return [p for p in selector_widget.value if p is not None]


def on_run_clicked(button):
    with output:
        output.clear_output(wait=True)

        if not dxf_chooser.value:
            print("Please select at least one DXF file.")
            return

        dxf_paths = get_selected_files(dxf_chooser)
        assembly_paths = get_selected_files(assembly_chooser)
        electrode_paths = get_selected_files(electrode_chooser)
        flex_pcb_paths = get_selected_files(flex_pcb_chooser)
        rigid_pcb_paths = get_selected_files(rigid_pcb_chooser)
        headstage_paths = get_selected_files(headstage_chooser)

        try:
            print("1. Building Assembly Router & tracing paths...")
            router = AssemblyRouter()

            component_files = [
                ("electrode", electrode_paths),
                ("flex_pcb", flex_pcb_paths),
                ("rigid_pcb", rigid_pcb_paths),
                ("headstage", headstage_paths),
            ]

            # The last selected non-electrode category is the terminal.
            active_non_electrode_categories = [
                category
                for category, paths in component_files
                if category != "electrode" and len(paths) > 0
            ]

            if not active_non_electrode_categories:
                print(
                    "Please select at least one target component "
                    "(flex_pcb, rigid_pcb, or headstage)."
                )
                return

            terminal_category = active_non_electrode_categories[-1]
            print(
                "   Terminal component category selected: "
                f"{terminal_category}"
            )

            instance_counts = defaultdict(int)
            component_categories = {}

            for category, paths in component_files:
                for path in paths:
                    definition = ComponentDefinition.from_json(path)

                    # Avoid registering identical models more than once.
                    if definition.model_name not in router.models:
                        router.register_model(definition)

                    instance_id = f"{category}_{instance_counts[category]}"
                    router.add_instance(instance_id, definition.model_name)

                    component_categories[instance_id] = category
                    instance_counts[category] += 1

            for path in assembly_paths:
                print(f"   Loading Assembly CSV: {path.name}")
                router.load_assembly_csv(path)

            starts = []
            electrode_instances = []
            electrode_channel_indices = {}

            for instance_id, category in component_categories.items():
                if category != "electrode":
                    continue

                electrode_instances.append(instance_id)

                model_name = router.instances[instance_id]
                definition = router.models[model_name]

                sorted_channels = sorted(
                    definition.mapping,
                    key=lambda channel: (
                        int(str(channel))
                        if str(channel).isdigit()
                        else str(channel)
                    ),
                )

                electrode_channel_indices[instance_id] = {
                    str(channel): index
                    for index, channel in enumerate(sorted_channels)
                }

                starts.extend(
                    f"{instance_id}.in_{channel}"
                    for channel in sorted_channels
                )

            report = router.report(
                starts,
                terminal_categories=[terminal_category],
                instance_categories=component_categories,
            )

            complete_report = [
                item for item in report if item["status"] == "complete"
            ]

            if not complete_report:
                print("No complete signal paths discovered by Assembly Router.")
                return

            electrode_instances.sort(key=lambda x: int(x.split("_")[1]))

            if len(dxf_paths) < len(electrode_instances):
                print(
                    f"Warning: Uploaded {len(dxf_paths)} DXF file(s) for "
                    f"{len(electrode_instances)} electrode instance(s). "
                    "Reusing the last DXF file for remaining electrodes."
                )

            print("2. Loading electrode DXF layouts...")
            electrode_xy_maps = {}
            electrode_offsets = {}
            running_offset = 0

            for index, instance_id in enumerate(electrode_instances):
                dxf_path = (
                    dxf_paths[index]
                    if index < len(dxf_paths)
                    else dxf_paths[-1]
                )
                print(f"   Mapping {instance_id} -> {dxf_path.name}")

                xy = utils.load_and_plot_electrodes(
                    dxf_path,
                    layer_name=layer_name,
                    num_contours=num_contours,
                )

                electrode_xy_maps[instance_id] = xy
                electrode_offsets[instance_id] = running_offset
                running_offset += len(xy)

            total_electrodes = running_offset

            print("3. Generating channel map matrix...")
            channel_map = utils.build_channel_map(
                complete_report,
                electrode_channel_indices,
                electrode_offsets,
                electrode_xy_maps,
                total_electrodes,
            )

            state["channel_map"] = channel_map
            state["dxf_file"] = str(dxf_paths[0])

            print("\nSuccess!")
            print(f"Total Electrodes Registered: {len(electrode_instances)}")
            print(f"Channel Map Shape: {channel_map.shape}")
            print(f"Sample column 0: {channel_map[:, 0]}")

        except Exception as error:
            print(f"Error generating channel map: {error}")

## Generate channel map

In [16]:
run_button.on_click(on_run_clicked)
display(run_button, output)

Button(button_style='success', description='Generate Channel Map', style=ButtonStyle())

Output()

In [9]:
export_button = widgets.Button(
    description="Export to CSV",
    button_style="info",
)
export_result = widgets.Output()


def export_channel_map(button):
    """Export the generated channel map to a CSV file."""
    with export_result:
        export_result.clear_output(wait=True)

        if state["channel_map"] is None:
            print("Generate a channel map first")
            return

        try:
            export_file = utils.export_channel_map_to_csv(
                state["channel_map"],
                state["dxf_file"],
            )
            print(f"Exported to: {export_file}")
        except Exception as error:
            print(f"Export failed: {error}")


export_button.on_click(export_channel_map)
display(export_button, export_result)


Button(button_style='info', description='Export to CSV', style=ButtonStyle())

Output()